# Сравнение моделей машинного обучения для предсказания цены автомобиля

В этом ноутбуке мы:
- Загрузим обработанные данные из `data/processed/`
- Обучим несколько моделей регрессии
- Проведём кросс-валидацию и подбор гиперпараметров
- Выберем лучшую модель и сохраним её

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.base import clone

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("Warning: XGBoost not installed")

try:
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print("Warning: CatBoost not installed")

sns.set_theme(style="whitegrid")

## Часть 1. Загрузка и подготовка данных

Загружаем файлы X_train, X_test, y_train, y_test из `data/processed/`.

In [ ]:
data_path = Path("../data/processed/")

X_train = pd.read_csv(data_path / "X_train.csv", index_col=0)
X_test = pd.read_csv(data_path / "X_test.csv", index_col=0)
y_train = pd.read_csv(data_path / "y_train.csv", index_col=0).squeeze()
y_test = pd.read_csv(data_path / "y_test.csv", index_col=0).squeeze()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
print("\nMissing values in X_train:")
print(X_train.isnull().sum())

print("\nMissing values in X_test:")
print(X_test.isnull().sum())

In [ ]:
y_train_prices = np.expm1(y_train)

print("\nСтатистика целевой переменной (в долларах):")
print(f"Среднее:     ${y_train_prices.mean():,.2f}")
print(f"Медиана:     ${y_train_prices.median():,.2f}")
print(f"Минимум:     ${y_train_prices.min():,.2f}")
print(f"Максимум:    ${y_train_prices.max():,.2f}")
print(f"Стд. отклонение: ${y_train_prices.std():,.2f}")

## Часть 2. Функции для оценки метрик

Функция `evaluate()` вычисляет MAE, RMSE, R² и MAPE в исходных единицах (долларах).

In [ ]:
def evaluate(name, model, X, y_log_true):
    y_pred_log = model.predict(X)
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_log_true)
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    
    print(f"\n{name}:")
    print(f"  MAE:  ${mae:,.2f}")
    print(f"  RMSE: ${rmse:,.2f}")
    print(f"  R²:   {r2:.4f}")
    print(f"  MAPE: {mape:.2f}%")
    
    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE": mape
    }

In [ ]:
def compare_models(results_list):
    df = pd.DataFrame(results_list)
    return df.sort_values("MAE").reset_index(drop=True)

## Часть 3. Определение моделей с дефолтными параметрами

In [ ]:
models = {
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
}

if XGB_AVAILABLE:
    models["XGBoost"] = XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)

if CATBOOST_AVAILABLE:
    models["CatBoost"] = CatBoostRegressor(iterations=100, verbose=0, random_state=42, thread_count=-1)

print(f"Total models defined: {len(models)}")
for name in models:
    print(f"  - {name}")

## Часть 4. Обучение всех моделей с дефолтными параметрами

In [ ]:
trained_models = {}
results = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    metrics = evaluate(name, model, X_test, y_test)
    results.append(metrics)

results_df = compare_models(results)
print("\n" + "="*60)
print("Сводная таблица результатов (отсортирована по MAE):")
print(results_df.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=results_df, x="MAE", y="Model", palette="viridis")
plt.title("Сравнение моделей по MAE")
plt.xlabel("MAE ($)")
plt.ylabel("")
plt.tight_layout()

save_path = Path("../reports/figures/model_comparison_mae.png")
save_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(save_path, dpi=150)
print(f"График сохранён в {save_path}")
plt.show()

## Часть 5. Кросс-валидация топ-3 моделей

In [ ]:
top3_names = results_df.head(3)["Model"].tolist()
print(f"Топ-3 модели: {top3_names}")

cv_results = {}

for name in top3_names:
    model = models[name]
    scores = cross_val_score(
        clone(model), X_train, y_train,
        cv=5, scoring="neg_mean_absolute_error", n_jobs=-1
    )
    mae_mean = -scores.mean()
    mae_std = scores.std()
    cv_results[name] = (mae_mean, mae_std)
    print(f"{name}: MAE = ${mae_mean:,.2f} ± ${mae_std:,.2f}")

In [ ]:
best_cv_model = min(cv_results, key=lambda x: cv_results[x][0])
print(f"\nНаиболее стабильная модель по кросс-валидации: {best_cv_model}")

## Часть 6. Подбор гиперпараметров

Тюним топ-2 модели по итогам кросс-валидации.

In [ ]:
top2_names = results_df.head(2)["Model"].tolist()
print(f"Тюнинг моделей: {top2_names}")

In [ ]:
param_distributions = {
    "RandomForest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"],
    },
    "GradientBoosting": {
        "n_estimators": [100, 200, 300, 500],
        "max_depth": [3, 4, 5, 6, 7, 8],
        "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
        "subsample": [0.7, 0.8, 0.9, 1.0],
    },
    "XGBoost": {
        "n_estimators": [100, 200, 300, 500],
        "max_depth": [3, 4, 5, 6, 7, 8],
        "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    },
    "CatBoost": {
        "iterations": [100, 200, 300],
        "depth": [4, 6, 8, 10],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "l2_leaf_reg": [1, 3, 5],
    },
    "DecisionTree": {
        "max_depth": [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
    }
}

In [ ]:
tuned_models = {}
tuning_results = []

for name in top2_names:
    print(f"\n=== Грубый поиск для {name} ===")
    base_model = models[name]
    param_dist = param_distributions.get(name, {})
    
    if not param_dist:
        print(f"  Нет параметров для тюнинга, пропускаем.")
        tuned_models[name] = base_model
        continue
    
    search = RandomizedSearchCV(
        base_model, param_dist,
        n_iter=20, cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1, random_state=42
    )
    search.fit(X_train, y_train)
    
    best_params = search.best_params_
    best_mae = -search.best_score_
    
    print(f"  Лучшие параметры: {best_params}")
    print(f"  MAE после грубого поиска: ${best_mae:,.2f}")
    
    tuned_models[name + "_coarse"] = search.best_estimator_

In [ ]:
for name in top2_names:
    coarse_name = name + "_coarse"
    if coarse_name not in tuned_models:
        continue
    
    print(f"\n=== Точный поиск для {name} ===")
    coarse_model = tuned_models[coarse_name]
    
    param_dist = param_distributions.get(name, {})
    narrowed_grid = {}
    
    for param, values in param_dist.items():
        current_val = getattr(coarse_model, param, None)
        if current_val is None and hasattr(coarse_model, 'get_params'):
            params_dict = coarse_model.get_params()
            current_val = params_dict.get(param)
        
        if isinstance(values, list) and current_val in values:
            idx = values.index(current_val)
            neighbors = values[max(0, idx-1):min(len(values), idx+2)]
            narrowed_grid[param] = neighbors
        else:
            narrowed_grid[param] = values[:3]
    
    grid_search = GridSearchCV(
        type(coarse_model)(), narrowed_grid,
        cv=5, scoring="neg_mean_absolute_error",
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    
    best_params = grid_search.best_params_
    best_mae = -grid_search.best_score_
    
    print(f"  Уточнённые параметры: {best_params}")
    print(f"  MAE после точного поиска: ${best_mae:,.2f}")
    
    tuned_models[name] = grid_search.best_estimator_

In [ ]:
default_mae = {row["Model"]: row["MAE"] for _, row in results_df.iterrows()}
tuned_mae = {}

comparison_data = []

for name in top2_names:
    if name in tuned_models:
        model = tuned_models[name]
        metrics = evaluate(name + " (tuned)", model, X_test, y_test)
        tuned_mae[name] = metrics["MAE"]
        
        improvement = (default_mae[name] - metrics["MAE"]) / default_mae[name] * 100
        comparison_data.append({
            "Model": name,
            "MAE_default": default_mae[name],
            "MAE_tuned": metrics["MAE"],
            "Improvement_%": improvement
        })

comparison_df = pd.DataFrame(comparison_data)
print("\nСравнение до и после тюнинга:")
print(comparison_df.to_string(index=False))

In [ ]:
if len(comparison_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(comparison_df))
    width = 0.35
    
    ax.bar(x - width/2, comparison_df["MAE_default"], width, label="Default", color="#1f77b4")
    ax.bar(x + width/2, comparison_df["MAE_tuned"], width, label="Tuned", color="#2ca02c")
    
    ax.set_ylabel("MAE ($)")
    ax.set_title("Сравнение MAE до и после подбора гиперпараметров")
    ax.set_xticks(x)
    ax.set_xticklabels(comparison_df["Model"])
    ax.legend()
    plt.tight_layout()
    
    save_path = Path("../reports/figures/tuning_comparison.png")
    plt.savefig(save_path, dpi=150)
    print(f"График сохранён в {save_path}")
    plt.show()

## Часть 7. Финальный ансамбль

Собираем VotingRegressor из тюненных версий топ-3 моделей.

In [ ]:
top3_for_ensemble = results_df.head(3)["Model"].tolist()
ensemble_estimators = []

for name in top3_for_ensemble:
    if name in tuned_models:
        ensemble_estimators.append((name, tuned_models[name]))
    elif name in models:
        ensemble_estimators.append((name, clone(models[name])))

if len(ensemble_estimators) >= 2:
    voting = VotingRegressor(estimators=ensemble_estimators)
    print("Обучение VotingRegressor...")
    voting.fit(X_train, y_train)
    
    metrics = evaluate("VotingEnsemble", voting, X_test, y_test)
    tuned_models["VotingEnsemble"] = voting
else:
    print("Недостаточно моделей для ансамбля")

## Часть 8. Выбор и сохранение финальной модели

In [ ]:
all_results = []

for name, model in tuned_models.items():
    metrics = evaluate(name, model, X_test, y_test)
    all_results.append(metrics)

final_df = compare_models(all_results)
print("\nФинальное сравнение всех моделей:")
print(final_df.to_string(index=False))

In [ ]:
best_model_name = final_df.iloc[0]["Model"]
best_model = tuned_models.get(best_model_name) or models.get(best_model_name.replace(" (tuned)", "").replace("_coarse", ""))

model_save_path = Path("../models/final_model.pkl")
model_save_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, model_save_path)
print(f"\nЛучшая модель сохранена в {model_save_path}")

feature_names = list(X_train.columns)
features_save_path = Path("../models/feature_names.pkl")
joblib.dump(feature_names, features_save_path)
print(f"Список признаков сохранён в {features_save_path}")

In [ ]:
best_row = final_df.iloc[0]
best_name = best_row["Model"]

default_name = best_name.replace(" (tuned)", "").replace("VotingEnsemble", "").replace("_coarse", "")
default_mae_val = default_mae.get(default_name, best_row["MAE"])
improvement = (default_mae_val - best_row["MAE"]) / default_mae_val * 100 if default_mae_val != best_row["MAE"] else 0

baseline_mae = np.expm1(y_train).std()
vs_baseline = (baseline_mae - best_row["MAE"]) / baseline_mae * 100

print("\n" + "="*60)
print("ИТОГОВЫЙ ВЫВОД")
print("="*60)
print(f"Выбранная модель: {best_name}")
print(f"MAE:  ${best_row['MAE']:,.2f}")
print(f"RMSE: ${best_row['RMSE']:,.2f}")
print(f"R²:   {best_row['R2']:.4f}")
print(f"MAPE: {best_row['MAPE']:.2f}%")
print(f"\nУлучшение vs дефолт: +{improvement:.1f}%")
print(f"Улучшение vs baseline: +{vs_baseline:.1f}%")
print("="*60)